[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C40_Research_Methodology_Course/05_writing_review/05_writing_review.ipynb)

# 05 · 科学写作与同行评审（动手做）

目标：把写作的诚实纪律跑成代码——**从结果生成带误差棒的诚实图表数据**、**claim-evidence 核对**、**审稿清单**、**可复现声明**。

路线：诚实图表数据(误差棒+判断重叠) → 截断坐标轴的欺骗 → claim-evidence 核对 → 审稿清单 → 可复现声明检查 → ✏️ 练习 → 📖 答案 → 🧪 真实论文报告格式胶囊。

> 核心心法：**主张不超过证据；图表不制造假象。** 诚实是科学写作的首要美德。

## 0 · 诚实图表自检清单(贯穿本 notebook)

本模块的核心是把『诚实』变成可执行的检查。每画一张图、每写一个主张，过一遍这张清单：

| 检查项 | 诚实做法 | 不诚实手法 |
|---|---|---|
| 误差棒 | 从真实多次运行算 std/SE/CI 并注明 | 干脆不画(隐藏不确定性) |
| 坐标轴 | y 轴从 0 或合理基线起 | 截断到 89-91 放大微小差异 |
| 基线 | 画出诚实的对照 | 只画自己的方法 |
| 样例 | 随机抽样 + 给失败案例 | 只晒精选成功案例 |
| 主张 | 每个主张指向具体证据，范围不超出 | 无证据的主张 / 过度声明 |

下面的 worked 例会把这张清单里的每一项都跑成代码。

## 1 · worked：从多种子结果生成诚实图表数据(带误差棒)

诚实图表的灵魂是误差棒，且必须从**真实多次运行**算出。我们聚合多种子结果，生成含均值+CI半宽的图表数据，并判断两个方法的误差棒是否**重叠**(重叠 -> 不能说『显著更好』)。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

# 两个方法各跑 10 个种子（A、B 真实水平几乎相同 -> 应得出『差异不可信』）
records = []
for method, true_acc in [('A', 0.900), ('B', 0.901)]:
    for seed in range(10):
        records.append(dict(method=method, seed=seed,
                            acc=true_acc + rng.normal(0, 0.006)))
df = pd.DataFrame(records)

def figure_data(df, value='acc', group='method', z=1.96):
    '''生成诚实图表数据：每组的 均值 与 误差棒半宽(用标准误*z 近似95%CI)。'''
    g = df.groupby(group)[value]
    out = g.agg(['mean', 'std', 'count']).reset_index()
    out['se'] = out['std'] / np.sqrt(out['count'])
    out['err'] = z * out['se']           # 误差棒半宽 ≈ 95% CI
    out['lo'] = out['mean'] - out['err']
    out['hi'] = out['mean'] + out['err']
    return out

fig = figure_data(df)
print(fig[['method','mean','err','lo','hi']].round(4).to_string(index=False))

def bars_overlap(fig, m1, m2):
    r1 = fig[fig.method == m1].iloc[0]
    r2 = fig[fig.method == m2].iloc[0]
    return not (r1['hi'] < r2['lo'] or r2['hi'] < r1['lo'])

overlap = bars_overlap(fig, 'A', 'B')
print(f'\nA 与 B 的误差棒重叠? {overlap}')
assert overlap, '本例 A、B 很接近，误差棒应重叠'
print('-> 结论：不能说『B 显著优于 A』。诚实的图必须让读者看到这种重叠。')
print('✅ 诚实图表数据：误差棒从真实多种子算出，并能判断差异是否可信。')

## 2 · worked：截断坐标轴如何制造『天壤之别』的假象

同一份数据：y 轴从 0 起 vs 截断到 89-91，视觉差异天壤之别。我们量化『截断把差异视觉放大了多少倍』。

In [ ]:
mean_a, mean_b = 89.8, 90.1     # 真实只差 0.3
diff = mean_b - mean_a

def visual_exaggeration(mean_a, mean_b, y_min, y_max):
    '''柱高差占坐标轴可见范围的比例 = 视觉上的『差异感』。'''
    axis_range = y_max - y_min
    return (mean_b - mean_a) / axis_range

honest = visual_exaggeration(mean_a, mean_b, 0, 100)      # y 轴 0-100
truncated = visual_exaggeration(mean_a, mean_b, 89, 91)   # y 轴截断到 89-91
print(f'真实差异 = {diff:.1f} 个百分点')
print(f'诚实坐标轴(0-100): 差异占可见范围 {honest:.1%}  (视觉上很小，符合实际)')
print(f'截断坐标轴(89-91): 差异占可见范围 {truncated:.1%}  (视觉上巨大!)')
print(f'截断把视觉差异放大了 {truncated/honest:.0f} 倍')
assert truncated / honest > 10, '截断坐标轴极大放大了视觉差异'
print('✅ 同一份数据，截断 y 轴把 0.3 分的差异视觉放大 50 倍 -> 经典视觉欺骗。')

## 3 · worked：claim-evidence 核对(检查自己的论文)

把每个主张和它的证据配对，自动揪出：无证据的主张、过度声明(范围超出证据)。

In [ ]:
# 论文的主张清单：每条含 主张文本、是否有证据、声称范围、证据覆盖范围
claims = [
    dict(text='B 在任务T上优于A', has_evidence=True,  claimed_scope=1, tested_scope=1),
    dict(text='B 适用于所有NLP任务', has_evidence=True,  claimed_scope=10, tested_scope=1),  # 过度!
    dict(text='B 更高效',           has_evidence=False, claimed_scope=1, tested_scope=0),  # 无证据!
]

def check_claims(claims):
    issues = []
    for cl in claims:
        if not cl['has_evidence']:
            issues.append((cl['text'], 'NO_EVIDENCE'))
        elif cl['claimed_scope'] > cl['tested_scope']:
            issues.append((cl['text'], 'OVERCLAIM'))
    return issues

issues = check_claims(claims)
print('claim-evidence 核对发现的问题：')
for text, kind in issues:
    print(f'  [{kind:12s}] {text}')
assert len(issues) == 2
assert ('B 更高效', 'NO_EVIDENCE') in issues
assert ('B 适用于所有NLP任务', 'OVERCLAIM') in issues
print('✅ 揪出：1 个无证据主张 + 1 个过度声明。投稿前先对自己审一遍。')

## 4 · worked：审稿清单——自动核验一篇论文的经验严谨性

把本课前四模块的检查表做成审稿清单：有对照？有消融？报方差？强基线？不过度声明？可复现？

In [ ]:
def review_checklist(paper):
    '''输入一篇论文的元信息(布尔)，返回审稿评级 + 必须修改项。'''
    checks = {
        '有对照组(模块02)':     paper.get('has_control', False),
        '有消融归因(模块02)':   paper.get('has_ablation', False),
        '报告方差(模块03)':     paper.get('reports_variance', False),
        '强基线(模块02)':       paper.get('strong_baseline', False),
        '不过度声明(模块05)':   paper.get('not_overclaimed', False),
        '可复现(模块04)':       paper.get('reproducible', False),
    }
    passed = sum(checks.values())
    must_fix = [k for k, v in checks.items() if not v]
    if passed == len(checks):     rating = 'accept'
    elif passed >= 4:             rating = 'major revision'
    else:                         rating = 'reject'
    return rating, must_fix

# 一篇『刷榜但不严谨』的论文
paper = dict(has_control=True, has_ablation=False, reports_variance=False,
             strong_baseline=True, not_overclaimed=False, reproducible=True)
rating, must_fix = review_checklist(paper)
print(f'审稿评级: {rating}')
print(f'必须修改: {must_fix}')
assert rating == 'reject', '缺3项核心严谨性 -> 拒'
assert '有消融归因(模块02)' in must_fix and '报告方差(模块03)' in must_fix
print('✅ 审稿清单把本课前四模块变成可操作的评审标准。')

---
## ✏️ 练习 1：误差棒数据(标准误)

实现 `error_bar(values, kind='se')`：给一组运行结果，返回误差棒半宽。
`kind='se'` 返回标准误 σ/√n；`kind='std'` 返回标准差 σ。(都用 ddof=1。)

In [ ]:
def error_bar(values, kind='se'):
    # TODO: v=np.asarray(values); std=v.std(ddof=1)
    #       kind=='se' -> std/sqrt(len(v))；kind=='std' -> std
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
vals = [90.0, 90.4, 89.6, 90.2, 89.8]
se = error_bar(vals, 'se')
std = error_bar(vals, 'std')
assert abs(std - np.std(vals, ddof=1)) < 1e-9
assert abs(se - np.std(vals, ddof=1)/np.sqrt(5)) < 1e-9
assert se < std, '标准误应小于标准差(除了√n)'
print('✅ 练习 1 通过：能算误差棒(SE 与 std)')

## ✏️ 练习 2：过度声明检测器

实现 `is_overclaim(claim)`：`claim` 是 dict，含 `claimed_scope`(声称范围) 与 `tested_scope`(验证范围)。
当声称范围 > 验证范围时返回 True(过度声明)。

In [ ]:
def is_overclaim(claim):
    # TODO: 返回 claim['claimed_scope'] > claim['tested_scope']
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert is_overclaim(dict(claimed_scope=10, tested_scope=1)) == True
assert is_overclaim(dict(claimed_scope=1, tested_scope=1)) == False
assert is_overclaim(dict(claimed_scope=1, tested_scope=3)) == False
print('✅ 练习 2 通过：能检测主张范围是否超出证据范围')

## ✏️ 练习 3：可复现声明检查

顶会要求论文附可复现信息。实现 `reproducibility_statement(paper)`：检查论文是否齐备
`['code', 'data', 'hyperparams', 'seeds', 'compute']` 五项，返回 (是否完整, 缺失项列表)。

In [ ]:
REQUIRED = ['code', 'data', 'hyperparams', 'seeds', 'compute']
def reproducibility_statement(paper):
    # TODO: paper 是 dict(项名->bool)。
    #   missing = [项 for 项 in REQUIRED if not paper.get(项, False)]
    #   返回 (len(missing)==0, missing)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
full = dict(code=True, data=True, hyperparams=True, seeds=True, compute=True)
partial = dict(code=True, data=True, hyperparams=False, seeds=False, compute=True)
ok, miss = reproducibility_statement(full)
assert ok == True and miss == []
ok2, miss2 = reproducibility_statement(partial)
assert ok2 == False
assert set(miss2) == {'hyperparams', 'seeds'}
print('✅ 练习 3 通过：能核验可复现声明是否齐备')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def error_bar(values, kind='se'):
    v = np.asarray(values)
    std = v.std(ddof=1)
    return std / np.sqrt(len(v)) if kind == 'se' else std

In [ ]:
# 练习 2 参考答案
def is_overclaim(claim):
    return claim['claimed_scope'] > claim['tested_scope']

In [ ]:
# 练习 3 参考答案
REQUIRED = ['code', 'data', 'hyperparams', 'seeds', 'compute']
def reproducibility_statement(paper):
    missing = [k for k in REQUIRED if not paper.get(k, False)]
    return (len(missing) == 0, missing)

---
## 🧪 真实数据胶囊：把结果写成顶会级的诚实报告行

顶会论文报告结果的标准格式是 `均值 ± 标准差 (n=种子数)`，并对最优值**加粗**、用统计检验标注显著性。
我们从真实风格的多种子结果，生成一个**可直接放进论文表格**的诚实报告——这是模块 03(统计)+04(聚合)+05(诚实)的集大成。

In [ ]:
rng = np.random.default_rng(2024)
# 三个方法的多种子结果(真实风格：B 最好，但与 C 接近)
data = []
for method, true_acc in [('baseline', 0.880), ('method_B', 0.905), ('method_C', 0.901)]:
    for seed in range(5):
        data.append(dict(method=method, acc=true_acc + rng.normal(0, 0.008)))
rdf = pd.DataFrame(data)

def paper_table(df, value='acc', group='method'):
    '''生成顶会级报告：均值±std (n)，最高均值加 ★ 标记。'''
    agg = df.groupby(group)[value].agg(['mean', 'std', 'count']).reset_index()
    best_mean = agg['mean'].max()
    lines = []
    for _, r in agg.sort_values('mean', ascending=False).iterrows():
        star = ' ★best' if abs(r['mean'] - best_mean) < 1e-12 else ''
        lines.append(f"{r[group]:12s}: {r['mean']*100:.1f} ± {r['std']*100:.1f} "
                     f"(n={int(r['count'])}){star}")
    return agg, lines

agg, lines = paper_table(rdf)
print('论文级结果表(均值±std 百分比, n=种子数)：')
for ln in lines: print('  ' + ln)
assert (agg['count'] == 5).all(), '每个方法 5 个种子'
assert agg.loc[agg['mean'].idxmax(), 'method'] == 'method_B', 'B 均值最高'
# 诚实性核验：B 和 C 很接近，需检查是否真的显著(用CI半宽近似)
b = rdf[rdf.method=='method_B']['acc']; cc = rdf[rdf.method=='method_C']['acc']
b_err = 1.96 * b.std(ddof=1)/np.sqrt(len(b))
c_err = 1.96 * cc.std(ddof=1)/np.sqrt(len(cc))
overlap = abs(b.mean() - cc.mean()) < (b_err + c_err)
print(f'\nB 与 C 的 95%CI 是否重叠: {overlap} -> {"不能宣称B显著优于C" if overlap else "B显著优于C"}')
print('✅ 顶会级诚实报告：均值±std±n，标最优，且诚实指出哪些差异其实不显著。')

**🧪 胶囊练习**：实现 `significant_winner(df, m1, m2, value='acc')`：判断 `m1` 是否**显著优于** `m2`。
准则：m1 均值更高 **且** 两者 95%CI(用 1.96*SE) 不重叠。返回 True/False。

In [ ]:
def significant_winner(df, m1, m2, value='acc', group='method'):
    # TODO: 取两组数据，各算 mean 与 95%CI半宽(1.96*std/sqrt(n))
    #   返回 (mean1>mean2) 且 (两区间不重叠: |mean1-mean2| > err1+err2)
    raise NotImplementedError

In [ ]:
# 自测
# B vs baseline 差距大 -> 应显著；B vs C 接近 -> 不显著
assert significant_winner(rdf, 'method_B', 'baseline') == True
assert significant_winner(rdf, 'method_B', 'method_C') == False
print('✅ 胶囊练习通过：能判断一个方法是否『显著』优于另一个(而非只看均值)')

In [ ]:
# 📖 胶囊参考答案
def significant_winner(df, m1, m2, value='acc', group='method'):
    def stats(m):
        v = df[df[group]==m][value]
        return v.mean(), 1.96 * v.std(ddof=1)/np.sqrt(len(v))
    mean1, err1 = stats(m1)
    mean2, err2 = stats(m2)
    return (mean1 > mean2) and (abs(mean1-mean2) > err1 + err2)

### 小结
- **写作=思考的一部分**；科学写作的首要美德是**诚实**(不误导)，不是优美。
- **IMRaD + claim-evidence 映射**：先写贡献列表当提纲与自检；每个主张都要能指向证据。
- **诚实图表**：必须有误差棒(从真实多次运行算)、标基线、坐标轴从合理起点；误差棒重叠=差异不可信。
- **同行评审**：审稿要严格而建设性；rebuttal 用证据回应；投稿前用前四模块检查表**预答辩**。
- **研究品味**：判断重要性与新颖性；理解/归因 > 刷榜(SOTA-chasing)；选题三问(主张/可证伪/证据)。

**全课完结**：读懂(01)→设计(02)→统计(03)→可复现(04)→诚实呈现(05)。把这套检查表变成做任何研究的默认安全带，你就从『会用模型的工程师』迈向了『能产出可信新知识的研究者』。